In [ ]:
# Install from github if running on Colab
!pip install git+https://github.com/maxiQE/fellowship_sim.git

# Ardeos — interactive sim

Run cells top-to-bottom once to initialise, then iterate on **Ability casts** + **Damage report**.

In [1]:
import random

from fellowship_sim import configure_logging
from fellowship_sim.ardeos import Talent
from fellowship_sim.ardeos.entity import Ardeos
from fellowship_sim.ardeos.setup import ArdeosSetup
from fellowship_sim.base_classes import Enemy, Legendary, State
from fellowship_sim.base_classes.stats import RawStatsFromScores

## General setup

In [2]:
# TRACE / DEBUG : verbose ability resolution
# INFO          : damage events
# SUCCESS       : important effects
# WARNING       : problems only
LOG_LEVEL = "INFO"
NUM_TARGETS = 5
SEED = 1234

configure_logging(LOG_LEVEL)

## Character setup

In [3]:
character_setup = ArdeosSetup(
    raw_stats=RawStatsFromScores(
        main_stat=2444.0,
        crit_score=900,
        expertise_score=1100,
        haste_score=1655,
        spirit_score=855,
    ),
    legendary=Legendary.NECK,
    talents=[
        Talent.BACKDRAFT,
        Talent.CRASH_AND_BURN,
        Talent.SLOW_BURN,
        Talent.UNDYING_FLAME,
        Talent.GREAT_BALLS_OF_FIRE,
        Talent.AGONIZING_BLAZE,
        Talent.ROLLING_FLAMES,
    ],
)

## Scenario

Call `reset_sim()` at the top of any ability sequence to start from a clean slate.

In [4]:
state: State
enemies: list[Enemy]
target: Enemy
ardeos: Ardeos

state = State(rng=random.Random(x=SEED))
enemies = [Enemy(state=state) for _ in range(NUM_TARGETS)]
target = enemies[0]
ardeos = character_setup.finalize(state)

def reset_sim() -> None:
    global state, enemies, target, ardeos
    state = State(rng=random.Random(x=SEED))
    enemies = [Enemy(state=state) for _ in range(NUM_TARGETS)]
    target = enemies[0]
    ardeos = character_setup.finalize(state)

reset_sim()

## Ability casts

In [5]:
reset_sim()

ardeos.engulfing_flames.cast(target)
ardeos.fire_ball.cast(target)
ardeos.searing_blaze.cast(target)
ardeos.incinerate.cast(target)
ardeos.searing_blaze.cast(target)
ardeos.fire_ball.cast(target)
ardeos.searing_blaze.cast(target)

SUCCESS  |    0.00 | Starting cast: Engulfing Flames
INFO     |    1.20 |   4982 dmg by engulfing_flames_dot(10.8s) on Enemy(13, dmg taken=4982)
SUCCESS  |    1.20 | Starting cast: Fire Ball
INFO     |    1.30 |  28388 dmg by Fire Ball on Enemy(13, dmg taken=33371)
INFO     |    1.30 |  35485 dmg by Fire Ball on Enemy(14, dmg taken=35485)
INFO     |    1.30 |  17743 dmg by Fire Ball on Enemy(15, dmg taken=17743)
INFO     |    1.30 |  17743 dmg by Fire Ball on Enemy(16, dmg taken=17743)
INFO     |    1.30 |  17743 dmg by Fire Ball on Enemy(17, dmg taken=17743)
INFO     |    2.40 |   4982 dmg by engulfing_flames_dot(9.6s) on Enemy(13, dmg taken=38353)
SUCCESS  |    2.40 | Starting cast: Searing Blaze
INFO     |    2.90 |   5678 dmg by fireball_dot(10.4s) on Enemy(13, dmg taken=44031)
INFO     |    2.90 |   7097 dmg by fireball_dot(10.4s) on Enemy(14, dmg taken=42583)
INFO     |    2.90 |   3549 dmg by fireball_dot(10.4s) on Enemy(15, dmg taken=21291)
INFO     |    2.90 |   3549 dmg by fi

<CastReturnCode.OK: 'ok'>

## Damage report

In [6]:
print("Total damage per enemy:")
for enemy in enemies:
    print(f"  Enemy {enemy.id}: {enemy.damage_tracker.total:.0f}")

print("\nMain target — by source:")
for source, record in sorted(target.damage_tracker.by_source.items(), key=lambda x: -x[1].total):
    print(f"  {source}: {record.total:.0f}")

Total damage per enemy:
  Enemy 13: 264842
  Enemy 14: 230321
  Enemy 15: 157502
  Enemy 16: 186643
  Enemy 17: 170879

Main target — by source:
  Incinerate: 87424
  FireBall: 56777
  EngulfingFlamesDoT: 54805
  FireBallDoT: 44192
  SearingBlazeDoT: 14672
  IncinerateDoT: 6973
